
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Guideline Judges with MLflow
## Overview
In this demo, we'll implement both global and per-row guideline judges. We'll define natural-language evaluation rules, run them against evaluation datasets, and inspect the results, including evaluating pre-generated outputs without calling the agent.

## Learning Objectives
By the end of this demo, you will be able to:
1. Implement the `Guidelines` judge for uniform evaluation criteria across all rows
2. Evaluate pre-generated inputs/outputs without a predict function
3. Apply `ExpectationsGuidelines` for per-row, scenario-specific evaluation

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;">This demo uses the agent created in <strong>Demo - Agent Setup</strong>. Please ensure you have completed that notebook before proceeding.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup
Run the following cell to configure your environment.

In [0]:
%run ../Includes/Classroom-Setup-3

In [0]:
print(f"Username:          {username}")
print(f"Catalog Name:      {catalog_name}")
print(f"Schema Name:       {schema_name}")

## B. Load Evaluation Datasets

We'll use three datasets stored in the UC volume, each demonstrating a different guideline evaluation pattern:
- **`guidelines_eval.json`** - Inputs only (agent generates outputs at evaluation time)
- **`guidelines_eval_pre_gen.json`** - Pre-generated inputs and outputs (no agent call needed)
- **`guidelines_eval_row_level.json`** - Includes per-row guidelines in the `expectations` field

In [0]:
import json
from pprint import pprint
from pathlib import Path

def read_eval_from_vol(file_name: str):
    path = Path(f"/Volumes/{catalog_name}/{schema_name}/agent_vol/{file_name}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

guidelines_dataset = read_eval_from_vol("guidelines_eval.json")
guidelines_dataset_pre_gen = read_eval_from_vol("guidelines_eval_pre_gen.json")
guidelines_dataset_row_level = read_eval_from_vol("guidelines_eval_row_level.json")


### B1. Inspect Dataset Structure

Let's examine the first dataset to understand the expected schema. Each entry contains an `inputs` field with the user's request: this is what gets passed to the agent during evaluation.

In [0]:
pprint(guidelines_dataset)

## C. Global Guidelines Evaluation

The `Guidelines` judge applies the same rules to every row. Here we create a guideline requiring responses in Spanish. Both inputs will fail since the agent responds in English.

In [0]:
from mlflow.genai.scorers import Guidelines

language_guideline = Guidelines(
    name="spanish",
    guidelines=["The response should be in Spanish"],
    model_name=guidelines_endpoint
)

### C1. Run Evaluation with Agent

In [0]:
guidelines_dataset_results = mlflow.genai.evaluate(
    data=guidelines_dataset,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[language_guideline]
)

### C2. Inspect Results

In [0]:
print(f"The run ID is: {guidelines_dataset_results.run_id}")
print(f"The aggregated metrics are: {guidelines_dataset_results.metrics}")
print("\nThe results from the previous batch of inputs:")
display(guidelines_dataset_results.result_df)

### C3. Evaluate Pre-Generated Outputs

When you already have inputs and outputs stored in a dataset, you can skip the `predict_fn` parameter entirely. The dataset `guidelines_eval_pre_gen.json` contains one correct Spanish response and one English response. Expect one pass and one fail.

In [0]:
guidelines_dataset_pre_gen_results = mlflow.genai.evaluate(
    data=guidelines_dataset_pre_gen,
    scorers=[language_guideline]
)


### C4. Inspect Pre-Generated Results

Since the dataset already contained outputs, the evaluation skipped agent invocation entirely. We expect one row to pass (Spanish response) and one to fail (English response).

In [0]:
print(f"The run ID is: {guidelines_dataset_pre_gen_results.run_id}")
print(f"The aggregated metrics are: {guidelines_dataset_pre_gen_results.metrics}")
print("\nThe results from the previous batch of inputs:")
display(guidelines_dataset_pre_gen_results.result_df)

## D. Per-Row Guidelines with ExpectationsGuidelines

`ExpectationsGuidelines` applies different guidelines to each row using the `expectations` field in the dataset. This is useful when different interactions require different evaluation criteria.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;"><code>ExpectationsGuidelines</code> requires an <code>outputs</code> field.</strong>
<p style="margin: 8px 0 0 0; color: #333;">They can be passed directly in the dataset or as a trace containing them. We will pass them directly.</p>
</div>
</div>
</div>


### D1. Define and Run Per-Row Evaluation

Unlike global `Guidelines`, `ExpectationsGuidelines` reads the `expectations` field from each row in the dataset. Each row can specify its own list of guidelines, enabling scenario-specific evaluation without separate evaluation runs.

In [0]:
from mlflow.genai.scorers import ExpectationsGuidelines

expected_guidelines = ExpectationsGuidelines(
    name="expected_guidelines",
    model_name=guidelines_endpoint
)

guidelines_dataset_row_level_results = mlflow.genai.evaluate(
    data=guidelines_dataset_row_level,
    scorers=[expected_guidelines]
)


### D2. Inspect Per-Row Results

Each row is scored against its own `expectations` guidelines. Rows meeting their specific criteria pass; those that don't will show which guideline was violated.

In [0]:
print(f"The run ID is: {guidelines_dataset_row_level_results.run_id}")
print(f"The aggregated metrics are: {guidelines_dataset_row_level_results.metrics}")
print("\nThe results from the previous batch of inputs:")
display(guidelines_dataset_row_level_results.result_df)

## E. Conclusion
In this demo, you:

1. Applied global `Guidelines` to evaluate all rows against uniform criteria (Spanish language requirement)
2. Evaluated pre-generated outputs without calling the agent by omitting `predict_fn`
3. Used `ExpectationsGuidelines` for per-row, scenario-specific evaluation criteria

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>

